# Figure 5 — hybrid power-grid disruption

Draft consolidated from the newly added `cascade.ipynb` and updated for the
final hybrid model:

- cell 32 draws the Category 1/2 length bars and impact lines;
- cell 40 draws the coastal network and hospital-density map.

The province summary is generated by
`prepare_hybrid_cascade_admin_summary.py`. In the legacy-compatible main-figure
convention, initial failure is directly removed transmission-line length and
cascade failure is surviving transmission length in completely unpowered
components. This matches the previous `cascade.ipynb` post-processing while
using the new hybrid service model. Partially served component length is not
included in these two bars. Simplified-edge lengths are sums over their full
unsimplified paths.


In [ ]:
import os
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(os.environ.get("INFRACASCADE_PROJECT_DIR", Path.cwd())).resolve()
if PROJECT_ROOT.name == "moz_notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = Path(os.environ.get("MOZ_FIGURE_DATA_ROOT", PROJECT_ROOT / "figure_data"))
FIGURE_DIR = Path(os.environ.get(
    "MOZ_MANUSCRIPT_FIGURE_DIR",
    PROJECT_ROOT / "outputs" / "manuscript_figures",
))
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Data:", DATA_ROOT)
print("Figures:", FIGURE_DIR)

import pickle
from shapely.geometry import shape


In [ ]:
ADMIN_SUMMARY_PATH = Path(os.environ.get(
    "INFRACASCADE_CASCADE_ADMIN_SUMMARY",
    DATA_ROOT / "cascade_admin_summary.gpkg",
))
POWER_GRAPH_PATH = Path(os.environ.get(
    "INFRACASCADE_POWER_GRAPH",
    PROJECT_ROOT / "network_save" / "simplified_hospital_MG.pickle",
))

admin = gpd.read_file(ADMIN_SUMMARY_PATH)
with POWER_GRAPH_PATH.open("rb") as handle:
    power_graph = pickle.load(handle)

NAME_COLUMN = "ADM1_NAME"
COASTAL_ORDER = [
    "Maputo", "Gaza", "Inhambane", "Sofala", "Zambezia", "Nampula", "Cabo Delgado"
]
required = {
    NAME_COLUMN,
    "num_hospital",
    "1_mean_initial_failure_length_km", "1_mean_cascade_failure_length_km",
    "1_cascade_initial_ratio", "1_mean_hospital_failure_probability",
    "2_mean_initial_failure_length_km", "2_mean_cascade_failure_length_km",
    "2_cascade_initial_ratio", "2_mean_hospital_failure_probability",
}
missing = required.difference(admin.columns)
if missing:
    raise ValueError(
        "Run prepare_cascade_admin_summary.py before executing this notebook. "
        f"Missing columns: {sorted(missing)}"
    )

coastal = admin[admin[NAME_COLUMN].isin(COASTAL_ORDER)].copy()
coastal[NAME_COLUMN] = pd.Categorical(
    coastal[NAME_COLUMN], categories=COASTAL_ORDER, ordered=True
)
coastal = coastal.sort_values(NAME_COLUMN)


In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, Patch, Rectangle
from matplotlib.ticker import FuncFormatter

fig = plt.figure(figsize=(8.333, 9.583), facecolor="white")
text_color = "#203040"
accent_color = "#14739B"

# Manuscript title and panel A reproduce the original composite layout.
fig.text(
    0.035, 0.963,
    "Category 1 and 2 cyclones can cause substantial electricity disruption",
    color=text_color, fontsize=11.5, va="top",
)
fig.add_artist(Line2D(
    [0.035, 0.965], [0.927, 0.927], transform=fig.transFigure,
    color="#B8C8D3", linewidth=0.6,
))
fig.text(0.035, 0.900, "A   Modeled disruption pathway", color=text_color, fontsize=8.5)

box_labels = (
    "Cyclone wind", "Grid damage", "Network disruption", "Facility electricity loss"
)
box_left = (0.035, 0.275, 0.515, 0.755)
box_width = 0.205
box_y = 0.795
box_height = 0.060
for index, (left, label) in enumerate(zip(box_left, box_labels)):
    fig.add_artist(Rectangle(
        (left, box_y), box_width, box_height,
        transform=fig.transFigure, facecolor="#EEF4F7",
        edgecolor=accent_color, linewidth=0.8,
    ))
    fig.text(
        left + box_width / 2, box_y + box_height / 2, label,
        color=text_color, fontsize=7.2, ha="center", va="center",
    )
    if index < len(box_left) - 1:
        fig.add_artist(FancyArrowPatch(
            (left + box_width, box_y + box_height / 2),
            (box_left[index + 1], box_y + box_height / 2),
            transform=fig.transFigure, arrowstyle="-|>",
            mutation_scale=7, color=accent_color, linewidth=0.7,
        ))

fig.text(
    0.035, 0.758,
    "Direct physical damage and subsequent loss of network supply are distinct outcomes.",
    color="#5A6B79", fontsize=7.2,
)
fig.text(
    0.035, 0.713, "B   Regional Category 1 / Category 2 results",
    color=text_color, fontsize=8.5,
)

map_axis = fig.add_axes([0.035, 0.095, 0.440, 0.555])
bar_axes = [
    fig.add_axes([0.530, 0.135, 0.190, 0.455]),
    fig.add_axes([0.755, 0.135, 0.190, 0.455]),
]

# Panel A: coastal provinces, hospital density, and power-grid edges.
coastal.plot(
    ax=map_axis,
    column="num_hospital",
    cmap="Greys",
    norm=Normalize(50, 200),
    edgecolor="black",
    linewidth=0.65,
)
for _, row in coastal.iterrows():
    point = row.geometry.representative_point()
    map_axis.text(
        point.x, point.y, str(row[NAME_COLUMN]),
        fontsize=5.7, ha="center", va="center", color="black",
    )

for edge in power_graph.edges:
    source, target = edge[:2]
    source_data = power_graph.nodes[source]
    target_data = power_graph.nodes[target]
    if source_data.get("power") == "hospital" or target_data.get("power") == "hospital":
        continue
    source_point = shape(source_data["geometry"])
    target_point = shape(target_data["geometry"])
    source_point = source_point if source_point.geom_type == "Point" else source_point.centroid
    target_point = target_point if target_point.geom_type == "Point" else target_point.centroid
    map_axis.plot(
        [source_point.x, target_point.x], [source_point.y, target_point.y],
        color="gold", linewidth=0.9, zorder=3,
    )

map_axis.set_xlim(30, 41)
map_axis.set_ylim(-27, -10)
map_axis.set_xticks([])
map_axis.set_yticks([])
map_axis.set_aspect("equal")
for spine in map_axis.spines.values():
    spine.set_visible(False)

map_axis.legend(
    handles=[Line2D([], [], color="gold", linewidth=1.2, label="power line")],
    loc="upper left", bbox_to_anchor=(0.10, 0.96), frameon=False,
    fontsize=5.5, handlelength=1.6,
)
colorbar_axis = map_axis.inset_axes([0.705, 0.055, 0.045, 0.475])
colorbar = fig.colorbar(
    ScalarMappable(norm=Normalize(50, 200), cmap="Greys"), cax=colorbar_axis
)
colorbar.set_ticks(np.arange(60, 201, 20))
colorbar.ax.tick_params(labelsize=5.5, length=2)
colorbar.outline.set_linewidth(0.5)
colorbar.set_label(
    "# Hospitals depending on public electrical grid", fontsize=6.2, labelpad=3
)

# Panels B/C: use the exact legacy notebook axes and styling.
positions = np.arange(len(coastal))
for category, axis, panel in zip((1, 2), bar_axes, ("(b)", "(c)")):
    initial = coastal[
        f"{category}_mean_initial_failure_length_km"
    ].to_numpy(float)
    cascade = coastal[
        f"{category}_mean_cascade_failure_length_km"
    ].to_numpy(float)
    ratio = coastal[f"{category}_cascade_initial_ratio"].to_numpy(float) * 100
    hospital_impact = coastal[
        f"{category}_mean_hospital_failure_probability"
    ].to_numpy(float) * 100

    axis.barh(positions, initial, color="#332288", height=0.30)
    axis.barh(
        positions, cascade, left=initial, color="#88CCEE", height=0.30
    )
    axis.set_xlim(0.0, 1000.0)
    axis.set_xticks(np.arange(0.0, 1001.0, 200.0))
    axis.set_xlabel("Failed Power Line Length (km)", fontsize=5.7, labelpad=2)
    axis.set_yticks(positions)
    axis.set_yticklabels(coastal[NAME_COLUMN] if category == 1 else [])
    if category == 1:
        plt.setp(axis.get_yticklabels(), rotation=45, ha="right")
    axis.tick_params(labelsize=5.5, width=0.5, length=2)
    axis.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.35)
    axis.set_title(f"TC Category {category}", fontsize=7.2, pad=25)
    for spine in axis.spines.values():
        spine.set_linewidth(0.5)

    percentage_axis = axis.twiny()
    percentage_axis.plot(
        ratio, positions, color="#D55E00", marker="o",
        linewidth=0.9, markersize=2.8,
    )
    percentage_axis.plot(
        hospital_impact, positions, color="#999933", marker="s",
        linewidth=0.9, markersize=2.8,
    )
    percentage_axis.set_xlim(0.0, 25.0)
    percentage_axis.set_xticks(np.arange(0.0, 25.1, 5.0))
    percentage_axis.xaxis.set_major_formatter(
        FuncFormatter(lambda value, _: f"{value:.0f}%")
    )
    percentage_axis.set_xlabel("Percentage", fontsize=6.2, labelpad=1)
    percentage_axis.tick_params(labelsize=5.5, width=0.5, length=2)
    for spine in percentage_axis.spines.values():
        spine.set_linewidth(0.5)

legend = [
    Patch(color="#332288", label="Initial Failure"),
    Patch(color="#88CCEE", label="Cascading Failure"),
    Line2D(
        [], [], color="#D55E00", marker="o", linewidth=0.9, markersize=2.8,
        label="Failure Ratio (Cascading / Initial) (%)",
    ),
    Line2D(
        [], [], color="#999933", marker="s", linewidth=0.9, markersize=2.8,
        label="Potential Hospital Impact (%)",
    ),
]
fig.legend(
    handles=legend, loc="upper left", bbox_to_anchor=(0.525, 0.680),
    ncol=2, frameon=False, fontsize=5.5, handlelength=1.8,
    columnspacing=1.0,
)

fig.text(0.245, 0.082, "(a)", fontsize=8, ha="center")
fig.text(0.625, 0.082, "(b)", fontsize=8, ha="center")
fig.text(0.850, 0.082, "(c)", fontsize=8, ha="center")
fig.text(
    0.035, 0.035,
    "Original regional map, axes and legends are retained; quantitative series use hybrid-model results.",
    color="#5A6B79", fontsize=6.3,
)
fig.text(
    0.035, 0.017,
    "Upper percentage axis: failure ratio and potentially affected facilities; lower axis: power-line length.",
    color="#5A6B79", fontsize=6.3,
)

for suffix in ("pdf", "png"):
    fig.savefig(FIGURE_DIR / f"figure_05.{suffix}", dpi=300)
plt.show()
